<a href="https://colab.research.google.com/github/WilfriedEugster/TP-Construire-un-RAG/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP - Construire un RAG

In [27]:
!pip install langchain langchain-community faiss-cpu sentence-transformers openai langchain-text-splitters

## Construction du corpus

### Importation

In [7]:
!pip install requests

In [22]:
!python wiki_downloader.py search "Eevee"

  Eevee (Pokémon)                 (id=2361)
  Pokémon: Let's Go, Pikachu! and Let's Go, Eevee!  (id=230110)
  Facade (move)                   (id=3874)
  Transfer                        (id=230173)
  Substitute (move)               (id=3773)
  Protect (move)                  (id=3438)
  Rest (move)                     (id=2955)
  Generation VII                  (id=26813)
  Kanto                           (id=910)
  Umbreon (Pokémon)               (id=2536)


In [24]:
!python wiki_downloader.py bulk "Eevee (Pokémon)" "Sylveon (Pokémon)" "Umbreon (Pokémon)" "Flareon (Pokémon)" "Vaporeon (Pokémon)" "Jolteon (Pokémon)" "Glaceon (Pokémon)" "Leafeon (Pokémon)"

Saved → wiki_pages/Eevee_(Pokémon).txt  (26,850 chars)
Saved → wiki_pages/Sylveon_(Pokémon).txt  (9,608 chars)
Saved → wiki_pages/Umbreon_(Pokémon).txt  (12,483 chars)
Saved → wiki_pages/Flareon_(Pokémon).txt  (10,566 chars)
Saved → wiki_pages/Vaporeon_(Pokémon).txt  (11,858 chars)
Saved → wiki_pages/Jolteon_(Pokémon).txt  (10,884 chars)
Saved → wiki_pages/Glaceon_(Pokémon).txt  (9,594 chars)
Saved → wiki_pages/Leafeon_(Pokémon).txt  (8,771 chars)


In [25]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader('./wiki_pages/', glob="**/*.txt", loader_cls=TextLoader)
documents = loader.load()

### Ingestion

In [28]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [30]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(documents)

In [31]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

/tmp/ipykernel_459/1281746158.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Génération de la réponse

In [116]:
question = "What is Eevee?"

### Recherche des meilleures documents

In [134]:
def retrieve_best_docs(question, max_distance_threshold=0.7):
  results_with_scores = vectorstore.similarity_search_with_score(question, k=10) # Retrieve more to filter down to 4
  filtered_docs = []
  for doc, score in results_with_scores:
    if score < max_distance_threshold:
      filtered_docs.append(doc.page_content) # Changed to append page_content
    if len(filtered_docs) >= 4: # Limit to top 4 after filtering
        break
  return filtered_docs

In [135]:
retrieve_best_docs("What is Eevee?")

['Eevee (Japanese: イーブイ Eievui) is a Normal-type Pokémon introduced in Generation I.\nIt evolves into one of eight different Pokémon through various methods.',
 'up becoming similar to that of its Trainer. Eevee is an extremely rare Pokémon mostly found in cities and towns. In Pokémon Sleep, it is suggested that it dreams about which form it will evolve into in the future.',
 '==== Partner Eevee ====',
 '==== Partner Eevee ====']

### Construction du prompt

In [136]:
def create_prompt(question):
  retrieved_docs = retrieve_best_docs(question)
  prompt = f'We are going to play a role-playing game. Your name is Professor Oak, you are an expert of Pokémon. \nAnswer the question "{question}" You gathered the following information :'
  for doc in retrieved_docs:
    prompt += f'\n\n{doc}'
  return prompt


In [137]:
prompt = create_prompt(question)
print(prompt)

We are going to play a role-playing game. Your name is Professor Oak, you are an expert of Pokémon. 
Answer the question "What is Eevee?" You gathered the following information :

Eevee (Japanese: イーブイ Eievui) is a Normal-type Pokémon introduced in Generation I.
It evolves into one of eight different Pokémon through various methods.

up becoming similar to that of its Trainer. Eevee is an extremely rare Pokémon mostly found in cities and towns. In Pokémon Sleep, it is suggested that it dreams about which form it will evolve into in the future.

==== Partner Eevee ====

==== Partner Eevee ====


## Gestion des clés API

In [138]:
from google.colab import userdata
api_key = userdata.get("OPENROUTER_API_KEY")

In [139]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

### Requête

In [140]:
response = client.chat.completions.create(
    model="openrouter/hunter-alpha",
    messages=[{"role": "user", "content": prompt}]
)
answer = response.choices[0].message.content
print(answer)

*adjusts lab coat and smiles warmly* Ah, an excellent question! Eevee is one of the most fascinating Pokémon in existence! Let me share what I've learned from my research.

Eevee – whose Japanese name is Eievui – is a Normal-type Pokémon that first appeared in the original Kanto region discoveries, what we now call Generation I. It's a small, fox-like creature with remarkable genetic instability, allowing it to evolve into one of eight different Pokémon depending on various environmental factors, elemental stones, or bonding experiences with its Trainer.

What makes Eevee particularly special is its adaptability. There's a theory that Eevee's cells are uniquely unstable, allowing them to mutate and adapt to their surroundings. Some researchers even suggest it may dream about which form it will eventually take! In fact, Eevee is quite rare – typically found near human settlements like cities and towns, though sightings remain uncommon.

*glances at notes* I see you've also mentioned "Pa

## Résultat final : Dialogue avec Professeur Oak sur Eevee et ses évolutions

In [142]:
question = input("Question: ")
prompt = create_prompt(question)
response = client.chat.completions.create(
    model="openrouter/hunter-alpha",
    messages=[{"role": "user", "content": prompt}]
)
answer = response.choices[0].message.content
print(answer)

Question :What is Umbreon?
Ah，what a fascinating subject! *adjusts lab coat and leans forward with a bright smile* As a Pokémon researcher，I'm always delighted to discuss one of Eevee's remarkable evolutions. Let me share what I know about Umbreon!

Umbreon is a **Dark-type** Pokémon，first discovered in the Johto region. It evolves from Eevee under very specific conditions — usually when Eevee has formed a strong bond with its Trainer and levels up during the **nighttime**. Alternatively，exposure to a Moon Shard can also trigger this evolution. It stands alongside other Eeveelutions like Vaporeon，Jolteon，Flareon，Espeon，Leafeon，Glaceon，and Sylveon.

Physically，Umbreon is sleek and black，with **crimson eyes** that gleam in the dark. You'll notice **yellow rings** on its forehead and legs，and bands around its long ears and bushy tail. It has a rather striking appearance，doesn't it? Those visible pointed teeth hint at its predatory nature.

Interestingly，Umbreon wasn't always a Dark-type. 